# RAG - Data ingestion to Vector DB pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\Mr. Mac\AppData\Local\Temp\ipykernel_5240\1867757213.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\Internship\Learn\Agentic AI\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    print(f"Found {len(pdf_files)} pdfs in the directory")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error : {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("data")
    

Found 5 pdfs in the directory
Processing: Computer Networking Notes for Tech Placements (1).pdf
Loaded 16 pages
Processing: DBMS_Notes (2).pdf
Loaded 27 pages
Processing: John doe_9 Dec 2025 - Copy (2).pdf
Loaded 1 pages
Processing: John doe_9 Dec 2025 - Copy.pdf
Loaded 1 pages
Processing: John doe_9 Dec 2025.pdf
Loaded 1 pages
Total documents loaded: 46


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m93 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Computer Networks.docx', 'source': 'data\\pdf\\Computer Networking Notes for Tech Placements (1).pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Computer Networking Notes for Tech Placements (1).pdf', 'file_type': 'pdf'}, page_content='Computer Networks\n●\nNetwork\n: A network is a set of devices that are connected\nwith a physical media link. In\na network, two or more nodes are connected by a physical\nlink or two or more networks\nare connected by one or more nodes. A network is a\ncollection of devices connected to\neach other to allow the sharing of data.\n●\nNetwork Topology\n:\nNetwork topology specifies the\nlayout of a computer network. It\nshows how devices and cables are connected to each\nother.\nTypes of Network Topology\n:\n●\nStar\n:\n●\nStar topology is a network topology in which all the\nnodes are connected\nto a single device 

In [6]:
### Text splitting into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunks:")
        print(f"content: {split_docs[0].page_content[200]}...")
        print(f"metadata: {split_docs[0].metadata}")


    return split_docs

In [7]:
chunks = split_documents(all_pdf_documents)
chunks

Split 46 into 83 chunks

Example chunks:
content: c...
metadata: {'producer': 'Skia/PDF m93 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Computer Networks.docx', 'source': 'data\\pdf\\Computer Networking Notes for Tech Placements (1).pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Computer Networking Notes for Tech Placements (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m93 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Computer Networks.docx', 'source': 'data\\pdf\\Computer Networking Notes for Tech Placements (1).pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Computer Networking Notes for Tech Placements (1).pdf', 'file_type': 'pdf'}, page_content='Computer Networks\n●\nNetwork\n: A network is a set of devices that are connected\nwith a physical media link. In\na network, two or more nodes are connected by a physical\nlink or two or more networks\nare connected by one or more nodes. A network is a\ncollection of devices connected to\neach other to allow the sharing of data.\n●\nNetwork Topology\n:\nNetwork topology specifies the\nlayout of a computer network. It\nshows how devices and cables are connected to each\nother.\nTypes of Network Topology\n:\n●\nStar\n:\n●\nStar topology is a network topology in which all the\nnodes are connected\nto a single device 

# Embedding and VectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()


    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 949.20it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Mr. Mac\AppData\Local\Temp\ipykernel_5240\3922915638.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [10]:
# Vector Store

In [15]:
class VectorStore:
    """Manages Vector embeddings in a chromadb vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store"):
        """Initialize the vector store
        Arg: 
        collection_name: Name of chromaDB collection
        persistent_directory: Directory to persist vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromadb client and collection"""
        try:
            # create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create a collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name, 
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any] , embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [16]:
chunks[:10]

[Document(metadata={'producer': 'Skia/PDF m93 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Computer Networks.docx', 'source': 'data\\pdf\\Computer Networking Notes for Tech Placements (1).pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Computer Networking Notes for Tech Placements (1).pdf', 'file_type': 'pdf'}, page_content='Computer Networks\n●\nNetwork\n: A network is a set of devices that are connected\nwith a physical media link. In\na network, two or more nodes are connected by a physical\nlink or two or more networks\nare connected by one or more nodes. A network is a\ncollection of devices connected to\neach other to allow the sharing of data.\n●\nNetwork Topology\n:\nNetwork topology specifies the\nlayout of a computer network. It\nshows how devices and cables are connected to each\nother.\nTypes of Network Topology\n:\n●\nStar\n:\n●\nStar topology is a network topology in which all the\nnodes are connected\nto a single device 

In [17]:
### convert text to embeddings
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
# store into the vector store
vector_store.add_documents(chunks, embeddings)



Generating embeddings for 83 texts...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.02s/it]


Generated embeddings with shape: (83, 384)
Adding 83 documents to vector store...
Successfully added 83 documents to vector store
Total documents in collection: 83


# Retriever pipeline from vector store

In [22]:
class RAGRetriver:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store:VectorStore,embedding_manager:EmbeddingManager ):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int=5,score_threshold:float=0.0 ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process_results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No Documents Found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriver(vector_store, embedding_manager)

In [23]:
rag_retriever

In [24]:

rag_retriever.retrieve("What is networking?")

Retrieving documents for query: 'What is networking?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.64it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_e0052d59_12',
  'content': '●\nError contro\nl\n: It detects and corrects the error occurred during the transmission from\nsource to destination.\n●\nAddressing\n:\nData-link layers attach the physical address\nwith the data frames so that\nthe individual machines can be easily identified.\n●\nLink management\n: Data-link layer manages the initiation,\nmaintenance and termination\nof the link between the source and destination for\nthe effective exchange of data.\n3.\nNetwork Layer\n●\nNetwork layer converts the logical address into the\nphysical address.\n●\nThe routing concept means it determines the best route\nfor the packet to travel from\nsource to the destination.\nFunctions of network layer\n:\n●\nRouting\n: The network layer determines the best route\nfrom source to destination. This\nfunction is known as routing.\n●\nLogical addressing\n: The network layer defines the\naddressing scheme to identify each\ndevice uniquely.\n●\nPacketizing\n: The network layer recei

# VectorDB to LLM output generation

In [27]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

from langchain.chat_models import init_chat_model
llm = init_chat_model(model="google_genai:gemini-flash-lite-latest")
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini Flash-Lite Latest', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-flash-lite-latest', client=<google.genai.client.Client object at 0x0000021414EBE660>, default_metadata=(), model_kwargs={})

In [34]:
def rag_simple(query, retriever, llm, top_k=3):
    # retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return "No relevant documents found."

    context = "\n\n".join(res['content'] for res in results) if results else ""
    if not context:
        return "No relevant context found."

    prompt = f"""
    Use the following context to answer the question concisely:
    {context}
    Question: {query}
    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [35]:
answer = rag_simple("What is networking?", rag_retriever, llm)

Retrieving documents for query: 'What is networking?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [43]:
answer[0]['text']

'Based on the provided context, a network is a set of devices (two or more nodes) connected by a physical media link (or two or more networks connected by one or more nodes) that allows for the sharing of data.'

In [45]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is IP address", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is IP address'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: [{'type': 'text', 'text': 'Based on the provided context, an IP address is a 32-bit dynamic address of a node in the network (specifically, an IPv4 address has 4 octets of 8-bit each with each number having a value up to 255). It is provided by an Internet Service Provider and is used to uniquely identify the connection of a network with a device taking part in that network.', 'extras': {'signature': 'El4KXAERTTIPvErV2uE+3KWel4z1NTTXr9wL3xzTLbgJMbRdCKRFkkLIBL8gChAcT3uWu1HUxLJUvtQVlN69X4KbDokQP+zLxp3BtViqEHzzYN88zzTuVdJL+pJFniAp'}}]
Sources: [{'source': 'Computer Networking Notes for Tech Placements (1).pdf', 'page': 4, 'score': 0.14016330242156982, 'preview': '●\nExtranet VPN:\nExtranet VPN uses shared infrastructure\nover an intranet, suppliers,\ncustomers, partners, and other entities and connects\nthem using dedicated connections.\n●\nIPv4 Address\n:\nAn IP address is a 32-bit dynamic address\nof a node in the network. An\nIPv4 address has 4 octets of 8-bit each ...'}, {'sou

In [48]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def _to_text(self, value):
        if value is None:
            return ""
        if isinstance(value, str):
            return value
        if isinstance(value, list):
            parts = []
            for item in value:
                if isinstance(item, dict):
                    if "text" in item:
                        parts.append(item["text"])
                    elif "content" in item:
                        parts.append(str(item["content"]))
                    else:
                        parts.append(str(item))
                else:
                    parts.append(str(item))
            return "".join(parts)
        return str(value)

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely.
                        Context:
                        {context}

                        Question: {question}

                        Answer:"""

            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke(prompt)
            answer = self._to_text(response.content)

        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = self._to_text(summary_resp.content)

        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is computer network?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is computer network?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 17.01it/s]

Generated embeddings with shape: (1, 384)


Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
                        Context:
                        Computer Networks
●
Network
: A network is a set of devices that are connected
with a physical media link. In
a network, two or more nodes are connected by a physical
link or two or more networks
are connected by one or more nodes. A network is a
collection of devices connected to
each other to allow the sharing of data.
●
Network Topology
:
Network topology specifies the
layout of a computer network. It
shows how devices and cables are connected to each
other.
Types of Network Topology
:
●
Star
:
●
Star topology is a network topology in which all the
nodes are connected
to a single device known as a central device.
●
Star topology requires more cable compared to other
topologies.
Therefore, it is more robust as a failure in one cable
will only disconnect a
specific computer connected to this cable.
●
If the centr